In [1]:
!pip install -q ortools osmnx geopandas networkx scikit-learn matplotlib shapely
print("Environment successfully prepared!")

Environment successfully prepared!


In [2]:
import os
import osmnx as ox
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point
import numpy as np
import networkx as nx

# Configure OSMnx settings
ox.settings.use_cache = True
ox.settings.log_console = False
ox.settings.overpass_endpoint = "https://overpass.kumi.systems/api/interpreter"

def get_yaba_network():
    west, south, east, north = 3.3650, 6.4950, 3.3900, 6.5250
    bbox = (west, south, east, north)
    print(f"Downloading street network for Yaba corridor...")
    try:
        G = ox.graph_from_bbox(bbox, network_type="drive")
        print(f"Real OSM Graph downloaded! Nodes: {len(G.nodes)}, Edges: {len(G.edges)}")
        return G
    except Exception as e:
        print(f"Notice: Overpass unreachable ({e}). Switching to Offline Synthetic Yaba Grid...")
        # Synthetic grid fallback
        G = nx.grid_2d_graph(8, 8, create_using=nx.MultiDiGraph)
        center_lat, center_lon = 6.5100, 3.3790
        lat_step, lon_step = 0.003, 0.003
        for r, c in G.nodes():
            G.nodes[(r, c)]['y'] = center_lat + (r - 4) * lat_step
            G.nodes[(r, c)]['x'] = center_lon + (c - 4) * lon_step
        mapping = {node: i for i, node in enumerate(G.nodes())}
        G = nx.relabel_nodes(G, mapping)
        for u, v, data in G.edges(data=True):
            data['length'] = 300.0
            data['travel_time'] = 36.0 # seconds
        return G

def extract_merchant_pois(G, n_merchants=40):
    node_ids = list(G.nodes())
    selected_nodes = np.random.choice(node_ids, size=min(n_merchants, len(node_ids)), replace=False)
    points = [Point(G.nodes[node]['x'], G.nodes[node]['y']) for node in selected_nodes]
    gdf_pois = gpd.GeoDataFrame(
        {'osmid': selected_nodes, 'shop': ['convenience'] * len(selected_nodes)},
        geometry=points,
        crs="EPSG:4326"
    )
    print(f"Loaded {len(gdf_pois)} merchant distribution targets.")
    return gdf_pois

# Execute network fetch & merchant generation
graph = get_yaba_network()
merchants = extract_merchant_pois(graph, n_merchants=45)

Real OSM Graph downloaded! Nodes: 791, Edges: 1807
Loaded 45 merchant distribution targets.


In [3]:
import pickle

def prepare_routing_graph(G):
    try:
        G = ox.routing.add_edge_speeds(G)
        G = ox.routing.add_edge_travel_times(G)
    except:
        for u, v, data in G.edges(data=True):
            if 'length' not in data: data['length'] = 300.0
            if 'travel_time' not in data: data['travel_time'] = 36.0
    return G

def compute_distance_time_matrices(G, merchants_gdf, depot_coords=(6.5100, 3.3790)):
    G = prepare_routing_graph(G)

    merchant_nodes = []
    for geom in merchants_gdf.geometry:
        pt = geom.centroid if geom.geom_type == 'Polygon' else geom
        node = ox.distance.nearest_nodes(G, X=pt.x, Y=pt.y)
        merchant_nodes.append(node)

    depot_node = ox.distance.nearest_nodes(G, X=depot_coords[1], Y=depot_coords[0])
    all_nodes = [depot_node] + merchant_nodes
    n_locations = len(all_nodes)

    dist_matrix = np.zeros((n_locations, n_locations))
    time_matrix = np.zeros((n_locations, n_locations))

    all_lengths_dist = dict(nx.all_pairs_dijkstra_path_length(G, weight='length'))
    all_lengths_time = dict(nx.all_pairs_dijkstra_path_length(G, weight='travel_time'))

    for i, u in enumerate(all_nodes):
        for j, v in enumerate(all_nodes):
            if u in all_lengths_dist and v in all_lengths_dist[u]:
                dist_matrix[i, j] = all_lengths_dist[u][v]
                time_matrix[i, j] = all_lengths_time[u][v]
            else:
                dist_matrix[i, j] = np.inf
                time_matrix[i, j] = np.inf

    os.makedirs("data/processed", exist_ok=True)
    routing_data = {
        "distance_matrix": dist_matrix,
        "duration_matrix": time_matrix,
        "nodes": all_nodes,
        "depot_node": depot_node
    }
    with open("data/processed/routing_matrices.pkl", "wb") as f:
        pickle.dump(routing_data, f)
    print("Matrices computed and saved successfully to data/processed/routing_matrices.pkl!")
    return dist_matrix, time_matrix, all_nodes

dist_mat, time_mat, all_nodes = compute_distance_time_matrices(graph, merchants)

/usr/local/lib/python3.13/dist-packages/osmnx/routing.py:553: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  edges[["highway", "speed_kph"]].set_index("highway").iloc[:, 0].fillna(hwy_speed_avg)


Matrices computed and saved successfully to data/processed/routing_matrices.pkl!


In [4]:
from ortools.constraint_solver import routing_enums_pb2, pywrapcp

def solve_cvrp():
    with open("data/processed/routing_matrices.pkl", "rb") as f:
        routing_data = pickle.load(f)

    dist_matrix = routing_data["distance_matrix"]
    int_distance_matrix = np.round(dist_matrix).astype(int).tolist()

    np.random.seed(42)
    n_locations = len(dist_matrix)
    demands = [0] + [int(np.random.randint(2, 10)) for _ in range(n_locations - 1)]

    num_vehicles = 3
    vehicle_capacity = 60

    manager = pywrapcp.RoutingIndexManager(len(int_distance_matrix), num_vehicles, 0)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        return int_distance_matrix[manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    def demand_callback(from_index):
        return demands[manager.IndexToNode(from_index)]

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index, 0, [vehicle_capacity] * num_vehicles, True, 'Capacity'
    )

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    search_parameters.time_limit.seconds = 3

    solution = routing.SolveWithParameters(search_parameters)

    if solution:
        print("\n=== OPTIMAL ROUTE DISPATCH PLAN (YABA) ===")
        total_distance = 0
        total_load = 0

        for vehicle_id in range(num_vehicles):
            index = routing.Start(vehicle_id)
            plan_output = f"Van {vehicle_id + 1}:\n"
            route_distance = 0
            route_load = 0

            while not routing.IsEnd(index):
                node_index = manager.IndexToNode(index)
                plan_output += f"  Node {node_index} (Demand: {demands[node_index]}) -> "
                previous_index = index
                index = solution.Value(routing.NextVar(index))
                route_distance += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)
                route_load += demands[node_index]

            node_index = manager.IndexToNode(index)
            plan_output += f"  Node {node_index} [Warehouse Return]\n"
            plan_output += f"  Distance: {route_distance / 1000.0:.2f} km | Load: {route_load} cartons\n"
            print(plan_output)
            total_distance += route_distance
            total_load += route_load

        print(f"--------------------------------------------------")
        print(f"Total Fleet Distance: {total_distance / 1000.0:.2f} km")
        print(f"Total Stock Distributed: {total_load} cartons.")

solve_cvrp()

/tmp/ipykernel_2344/2727909143.py:8: RuntimeWarning: invalid value encountered in cast
  int_distance_matrix = np.round(dist_matrix).astype(int).tolist()


In [5]:
from google.colab import files
files.download('data/processed/routing_matrices.pkl')
print("routing_matrices.pkl downloaded! Save it in your local project folder under data/processed/.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

routing_matrices.pkl downloaded! Save it in your local project folder under data/processed/.


In [6]:
!pip install -q simpy
print("SimPy installed successfully!")

SimPy installed successfully!


In [7]:
import simpy
import random
import numpy as np
import pickle
import os

class LagosDeliverySimulation:
    def __init__(self, env, vehicle_routes, duration_matrix, demands):
        self.env = env
        self.vehicle_routes = vehicle_routes  # List of routes from OR-Tools
        self.duration_matrix = duration_matrix  # Base travel times in seconds
        self.demands = demands

        self.metrics = {
            "completed_deliveries": 0,
            "failed_deliveries": 0,
            "total_delay_seconds": 0,
            "vehicle_breakdowns": 0
        }

    def run_fleet(self):
        sim_vehicles = [
            self.env.process(self.vehicle_trip(v_id, route))
            for v_id, route in enumerate(self.vehicle_routes)
        ]
        yield self.env.all_of(sim_vehicles)

    def vehicle_trip(self, vehicle_id, route):
        print(f"Van {vehicle_id + 1} departing warehouse at t={self.env.now:.1f} min.")

        current_node = route[0]  # Depot
        for next_node in route[1:]:
            base_time = self.duration_matrix[current_node][next_node]

            # Stochastic Lagos traffic multiplier (Gamma distribution)
            traffic_multiplier = np.random.gamma(shape=2.0, scale=0.6) + 1.0
            effective_travel_time = (base_time / 60.0) * traffic_multiplier  # minutes

            # 2% chance of vehicle breakdown en route
            if random.random() < 0.02:
                breakdown_delay = random.uniform(15, 45)
                print(f"  [BREAKDOWN] Van {vehicle_id + 1} broke down! Delay: +{breakdown_delay:.1f} mins")
                self.metrics["vehicle_breakdowns"] += 1
                yield self.env.timeout(breakdown_delay)
                self.metrics["total_delay_seconds"] += breakdown_delay * 60

            yield self.env.timeout(effective_travel_time)

            if next_node == 0:
                print(f"Van {vehicle_id + 1} returned to warehouse at t={self.env.now:.1f} min.")
                break

            demand = self.demands[next_node]
            service_time = demand * 1.5  # 1.5 mins per carton offloaded

            # 5% chance merchant rejects delivery (cash flow constraints)
            if random.random() < 0.05:
                print(f"  [REJECTION] Merchant at Node {next_node} rejected delivery.")
                self.metrics["failed_deliveries"] += 1
                yield self.env.timeout(5.0)
            else:
                yield self.env.timeout(service_time)
                self.metrics["completed_deliveries"] += 1

            current_node = next_node

def run_simulation_experiment():
    if not os.path.exists("data/processed/routing_matrices.pkl"):
        print("Error: Please run your Distance Matrix & VRP cells first so routing_matrices.pkl exists!")
        return

    with open("data/processed/routing_matrices.pkl", "rb") as f:
        data = pickle.load(f)

    duration_matrix = data["duration_matrix"]

    # Sample routes using nodes from your generated graph
    n_nodes = len(duration_matrix)
    sample_routes = [
        [0, 1, 2, 3, 0],
        [0, 4, 5, 6, 0],
        [0, 7, 8, 9, 0]
    ]
    # Ensure route node indices are within bounds
    sample_routes = [[node for node in route if node < n_nodes] for route in sample_routes]
    for r in sample_routes:
        if len(r) > 2 and r[-1] != 0: r.append(0)
        if len(r) <= 2: r = [0, 0]

    demands = {i: random.randint(2, 10) for i in range(n_nodes)}
    demands[0] = 0

    env = simpy.Environment()
    sim_model = LagosDeliverySimulation(env, sample_routes, duration_matrix, demands)

    print("=== STARTING LAGOS OPERATIONAL CHAOS SIMULATION ===")
    env.process(sim_model.run_fleet())
    env.run()

    print("\n=== SIMULATION SUMMARY REPORT ===")
    print(f"Completed Deliveries : {sim_model.metrics['completed_deliveries']}")
    print(f"Failed / Rejected    : {sim_model.metrics['failed_deliveries']}")
    print(f"Vehicle Breakdowns   : {sim_model.metrics['vehicle_breakdowns']}")
    print(f"Total Extra Delays   : {sim_model.metrics['total_delay_seconds'] / 60.0:.1f} minutes")

run_simulation_experiment()

=== STARTING LAGOS OPERATIONAL CHAOS SIMULATION ===
Van 1 departing warehouse at t=0.0 min.
Van 2 departing warehouse at t=0.0 min.
Van 3 departing warehouse at t=0.0 min.
Van 2 returned to warehouse at t=111.9 min.
Van 3 returned to warehouse at t=190.5 min.
Van 1 returned to warehouse at t=inf min.

=== SIMULATION SUMMARY REPORT ===
Completed Deliveries : 9
Failed / Rejected    : 0
Vehicle Breakdowns   : 0
Total Extra Delays   : 0.0 minutes


In [8]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import pickle
import os

def generate_simulated_training_data(n_samples=5000):
    """
    Simulates historical delivery logs capturing environmental chaos
    similar to your SimPy simulation engine.
    """
    np.random.seed(42)

    # 1. Feature Engineering
    traffic_multiplier = np.random.gamma(shape=2.0, scale=0.6, size=n_samples) + 1.0
    distance_to_depot = np.random.uniform(1.0, 15.0, size=n_samples)  # in km
    demand_cartons = np.random.randint(2, 12, size=n_samples)
    route_position = np.random.randint(1, 8, size=n_samples)
    is_peak_hour = np.random.choice([0, 1], size=n_samples, p=[0.6, 0.4])

    # 2. Latent Failure Probability Logic (Ground Truth Mapping)
    # Failure risk spikes with high traffic, peak hours, and greater distance from depot
    failure_logit = (
        -3.2
        + 1.3 * (traffic_multiplier - 1.0)
        + 0.12 * distance_to_depot
        + 0.9 * is_peak_hour
        + 0.08 * route_position
        + 0.05 * demand_cartons
    )

    # Convert logit to probability via sigmoid function
    prob_failure = 1 / (1 + np.exp(-failure_logit))
    failure_target = (np.random.random(n_samples) < prob_failure).astype(int)

    # 3. Assemble Dataset
    df = pd.DataFrame({
        'traffic_multiplier': traffic_multiplier,
        'distance_to_depot': distance_to_depot,
        'demand_cartons': demand_cartons,
        'route_position': route_position,
        'is_peak_hour': is_peak_hour,
        'delivery_failed': failure_target
    })
    return df

def train_xgboost_model():
    print("Generating training dataset from simulation patterns...")
    df = generate_simulated_training_data(n_samples=5000)

    X = df.drop(columns=['delivery_failed'])
    y = df['delivery_failed']

    # Train/Test Split (stratified to preserve failure class balance)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    print(f"Training samples: {X_train.shape[0]} | Baseline Failure Rate: {y.mean()*100:.1f}%")

    # Initialize XGBoost Classifier optimized for tabular risk scoring
    model = xgb.XGBClassifier(
        n_estimators=150,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    print("Training XGBoost Delivery Risk Predictor...")
    model.fit(X_train, y_train)

    # Evaluate Performance
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print("\n=== MODEL PERFORMANCE EVALUATION ===")
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}")

    # Feature Importance Breakdown
    importance = pd.DataFrame({
        'Feature': X.columns,
        'Importance': model.feature_importances_
    }).sort_values(by='Importance', ascending=False)

    print("\n=== TOP FAILURE RISK DRIVERS ===")
    print(importance.to_string(index=False))

    # Save the trained model artifact for Phase 4 Dashboard inference
    os.makedirs("data/models", exist_ok=True)
    model_path = "data/models/risk_xgb_model.pkl"
    with open(model_path, "wb") as f:
        pickle.dump(model, f)
    print(f"\nTrained risk model successfully saved to {model_path}!")

if __name__ == "__main__":
    train_xgboost_model()

Generating training dataset from simulation patterns...
Training samples: 4000 | Baseline Failure Rate: 55.7%
Training XGBoost Delivery Risk Predictor...

=== MODEL PERFORMANCE EVALUATION ===
              precision    recall  f1-score   support

           0       0.64      0.63      0.63       443
           1       0.71      0.72      0.72       557

    accuracy                           0.68      1000
   macro avg       0.68      0.67      0.67      1000
weighted avg       0.68      0.68      0.68      1000

ROC-AUC Score: 0.7624

=== TOP FAILURE RISK DRIVERS ===
           Feature  Importance
traffic_multiplier    0.376487
      is_peak_hour    0.235451
 distance_to_depot    0.172463
    demand_cartons    0.114948
    route_position    0.100652

Trained risk model successfully saved to data/models/risk_xgb_model.pkl!


In [9]:
!zip -r data.zip /content/data

  adding: content/data/ (stored 0%)
  adding: content/data/models/ (stored 0%)
  adding: content/data/models/risk_xgb_model.pkl (deflated 75%)
  adding: content/data/processed/ (stored 0%)
  adding: content/data/processed/routing_matrices.pkl (deflated 45%)
